# 02 · Storage: raw contracts → DuckDB

Loads `data/raw/contracts_<slug>.json` (written by `01_collection.ipynb`) into a small relational schema in a single file, `data/procurement.duckdb`.

| table | one row per | key |
|---|---|---|
| `contracts` | contract (includes the untouched raw record in `raw_record`) | `contract_id` |
| `entities` | company or public body | `nif` |
| `contract_parties` | contract × entity × role (`authority`, `contractor`, `competitor`) | — |
| `contract_cpvs` | contract × CPV code | — |
| `contract_locations` | contract × execution location | — |

Entities are keyed by **NIF**, not by name. The same body is often spelled several ways, for example
*"Unidade Local de Saúde da Cova da Beira, E. P. E."* and *"…, EPE"*, and those would otherwise count as different parties.

Loading is an **upsert** on `contract_id`, so re-running it every day never creates duplicates.

**Runs in Google Colab or locally.** The first code cell detects the environment. In Colab it mounts Google Drive and keeps everything under `My Drive/procurement-poc/data/`, so results survive the session.

In [1]:
# ── Environment: Google Colab or local (VS Code / Jupyter) ─────────────────────────────
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_FOLDER = "procurement-poc"   # folder in "My Drive" that holds the project data
PIP_PACKAGES = ["duckdb>=1.1"]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "data"   # persistent: survives the session
    WORK = Path("/content/work")                                     # fast local disk: wiped when the session ends
    if PIP_PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)
else:
    DATA = Path("data")
    WORK = DATA

DATA.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print(f"running in Colab: {IN_COLAB} | data folder: {DATA}")


# DuckDB is slow and can corrupt on a network drive, so in Colab the notebook works on a local copy
# of the database and copies it back to Drive with push_db(). Locally both paths are the same file.
DB_NAME = "procurement.duckdb"
DB_FILE = WORK / DB_NAME


def pull_db():
    """Colab: copy the database from Drive to local disk before connecting."""
    src = DATA / DB_NAME
    if WORK != DATA and src.exists():
        shutil.copy2(src, DB_FILE)
        print(f"database copied from Drive ({src.stat().st_size / 1e6:.1f} MB)")


def push_db(con=None):
    """Colab: save the local copy back to Drive. With an open connection, it checkpoints first."""
    if WORK == DATA:
        return
    if con is not None:
        con.execute("CHECKPOINT")
    tmp = DATA / (DB_NAME + ".tmp")
    shutil.copy2(DB_FILE, tmp)
    tmp.replace(DATA / DB_NAME)
    print(f"database saved to Drive ({DB_FILE.stat().st_size / 1e6:.1f} MB)")


Mounted at /content/drive
running in Colab: True | data folder: /content/drive/MyDrive/procurement-poc/data


In [2]:
import html
import json
import re

import duckdb
import pandas as pd

SLUG = "covilha"
RAW_FILE = DATA / "raw" / f"contracts_{SLUG}.json"

raw = json.loads(RAW_FILE.read_text(encoding="utf-8"))
print(f"{len(raw):,} raw contracts")

1,327 raw contracts


## 1. Schema

In [3]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS contracts (
    contract_id                BIGINT PRIMARY KEY,
    procedure_id               BIGINT,
    procedure_type             VARCHAR,
    contract_types             VARCHAR,
    object                     VARCHAR,
    description                VARCHAR,
    announcement_number        VARCHAR,
    publication_date           DATE,
    award_decision_date        DATE,
    signing_date               DATE,
    close_date                 DATE,
    base_price                 DECIMAL(18, 2),
    contract_price             DECIMAL(18, 2),
    effective_price            DECIMAL(18, 2),
    execution_days             INTEGER,
    legal_basis                VARCHAR,
    direct_award_justification VARCHAR,
    regime                     VARCHAR,
    award_criteria             VARCHAR,
    framework_agreement        VARCHAR,
    centralized                BOOLEAN,
    green_contract             BOOLEAN,
    documents_link             VARCHAR,
    notes                      VARCHAR,
    year                       INTEGER,
    raw_record                 VARCHAR,
    loaded_at                  TIMESTAMP
);
CREATE TABLE IF NOT EXISTS entities (
    nif  VARCHAR PRIMARY KEY,
    name VARCHAR
);
CREATE TABLE IF NOT EXISTS contract_parties (
    contract_id BIGINT,
    nif         VARCHAR,
    role        VARCHAR,   -- authority | contractor | competitor
    is_sme      BOOLEAN    -- set for contractors only
);
CREATE TABLE IF NOT EXISTS contract_cpvs (
    contract_id     BIGINT,
    cpv_code        VARCHAR,
    cpv_description VARCHAR
);
CREATE TABLE IF NOT EXISTS contract_locations (
    contract_id  BIGINT,
    country      VARCHAR,
    district     VARCHAR,
    municipality VARCHAR
);
-- one row per contract; joint buyers / consortium members are joined with ' + '
CREATE OR REPLACE VIEW v_contracts AS
WITH names AS (
    SELECT p.contract_id, p.role,
           string_agg(p.nif, ' + ' ORDER BY p.nif)   AS nifs,
           string_agg(e.name, ' + ' ORDER BY p.nif)  AS names
    FROM contract_parties p JOIN entities e USING (nif)
    WHERE p.role IN ('authority', 'contractor')
    GROUP BY ALL
)
SELECT c.contract_id, c.publication_date, c.procedure_type, c.object, c.contract_price,
       a.nifs AS authority_nif,  a.names AS authority,
       k.nifs AS contractor_nif, k.names AS contractor
FROM contracts c
LEFT JOIN names a ON a.contract_id = c.contract_id AND a.role = 'authority'
LEFT JOIN names k ON k.contract_id = c.contract_id AND k.role = 'contractor';
"""

pull_db()
con = duckdb.connect(str(DB_FILE))
con.execute(SCHEMA)

## 2. Parse the raw records into the table shapes
The raw fields pack several values into one string, so a few small helpers split them:
- parties: `"506361659 - Município da Covilhã"`, where bidders have no spaces around the dash and their names are double-HTML-escaped (`&amp;amp;`)
- CPVs: `"45233140-2 - Obras de estradas"`
- locations: `"Portugal, Castelo Branco, Covilhã"`
- dates are `dd/mm/yyyy`; yes/no flags are `"Sim"` / `"Não"`

In [4]:
PARTY_RE = re.compile(r"^\s*([^\s-]+)\s*-\s*(.+?)\s*$")
CPV_RE = re.compile(r"^\s*(\d{8}-\d)\s*-\s*(.*?)\s*$")


def split_party(text):
    """'NIF - Name' -> (nif, name). Entries without a NIF get a name-based key so they aren't dropped."""
    text = html.unescape(html.unescape(text))
    m = PARTY_RE.match(text)
    return (m.group(1), m.group(2)) if m else (f"NO-NIF:{text.strip()}", text.strip())


def split_cpv(text):
    m = CPV_RE.match(text)
    return (m.group(1), m.group(2)) if m else (None, text.strip())


def to_date(text):
    return pd.to_datetime(text, format="%d/%m/%Y", errors="coerce") if text else None


def yes(text):
    return {"Sim": True, "Não": False}.get(text)


def empty_to_none(text):
    return text.strip() if isinstance(text, str) and text.strip() else None

In [5]:
def parse(records):
    contracts, parties, cpvs, locations = [], [], [], []
    loaded_at = pd.Timestamp.now()

    for r in records:
        cid = int(r["idcontrato"])
        contracts.append({
            "contract_id": cid,
            "procedure_id": r["idprocedimento"],
            "procedure_type": r["tipoprocedimento"],
            "contract_types": "; ".join(r["tipoContrato"] or []),
            "object": empty_to_none(r["objectoContrato"]),
            "description": empty_to_none(r["descContrato"]),
            "announcement_number": empty_to_none(r["nAnuncio"]),
            "publication_date": to_date(r["dataPublicacao"]),
            "award_decision_date": to_date(r["dataDecisaoAdjudicacao"]),
            "signing_date": to_date(r["dataCelebracaoContrato"]),
            "close_date": to_date(r["dataFechoContrato"]),
            "base_price": r["precoBaseProcedimento"],
            "contract_price": r["precoContratual"],
            "effective_price": r["PrecoTotalEfetivo"],
            "execution_days": r["prazoExecucao"],
            "legal_basis": empty_to_none(r["fundamentacao"]),
            "direct_award_justification": empty_to_none(r["fundamentAjusteDireto"]),
            "regime": empty_to_none(r["regime"]),
            "award_criteria": empty_to_none(r["TipoCriterioAdjudicacao"]),
            "framework_agreement": empty_to_none(r["numAcordoQuadro"]),
            "centralized": yes(r["ProcedimentoCentralizado"]),
            "green_contract": yes(r["ContratEcologico"]),
            "documents_link": empty_to_none(r["linkPecasProc"]),
            "notes": empty_to_none(r["Observacoes"]),
            "year": r["Ano"],
            "raw_record": json.dumps(r, ensure_ascii=False),
            "loaded_at": loaded_at,
        })

        smes = set(r["adjudicatarioPMEs"] or [])
        for role, field in [("authority", "adjudicante"), ("contractor", "adjudicatarios"), ("competitor", "concorrentes")]:
            for text in r[field] or []:
                nif, name = split_party(text)
                parties.append({"contract_id": cid, "nif": nif, "name": name, "role": role,
                                "is_sme": (nif in smes) if role == "contractor" else None})

        for text in r["cpv"] or []:
            code, desc = split_cpv(text)
            cpvs.append({"contract_id": cid, "cpv_code": code, "cpv_description": desc})

        for text in r["localExecucao"] or []:
            country, district, municipality = (text.split(", ", 2) + [None, None])[:3]
            locations.append({"contract_id": cid, "country": country, "district": district, "municipality": municipality})

    parties = pd.DataFrame(parties).drop_duplicates(["contract_id", "nif", "role"])
    entities = parties[["nif", "name"]].drop_duplicates("nif", keep="last")  # the latest spelling wins
    return {
        "contracts": pd.DataFrame(contracts),
        "entities": entities,
        "contract_parties": parties.drop(columns="name"),
        "contract_cpvs": pd.DataFrame(cpvs).drop_duplicates(),
        "contract_locations": pd.DataFrame(locations).drop_duplicates(),
    }


tables = parse(raw)
{name: len(df) for name, df in tables.items()}

{'contracts': 1327,
 'entities': 966,
 'contract_parties': 4045,
 'contract_cpvs': 1328,
 'contract_locations': 1438}

In [6]:
tables["contract_parties"].merge(tables["entities"], on="nif").head(6)

,contract_id,nif,role,is_sme,name
0,12446543,505330768,authority,None,Município da Covilhã
1,12446543,501371974,contractor,True,"Amadeu Gonçalves Cura & Filhos, Lda."
2,12446543,503335096,competitor,None,"Irmãos Almeida Cabral, Lda"
3,12446543,518554589,competitor,None,"SOCOBI - Construções do Interior, Unipessoal Lda."
4,12446543,504504754,competitor,None,"Betovial - Engenharia e Construções, Lda."
5,12446543,501371974,competitor,None,"Amadeu Gonçalves Cura & Filhos, Lda."


## 3. Upsert
All inside one transaction:
1. entities: `INSERT OR REPLACE` on `nif`, so a new spelling of a name overwrites the old one
2. every contract in the batch: remove its old rows from `contracts` and the child tables, then insert the fresh ones

Running the same batch twice leaves the tables exactly as they were.

In [7]:
CHILD_TABLES = ["contract_parties", "contract_cpvs", "contract_locations"]


def upsert(con, tables):
    con.execute("BEGIN TRANSACTION")
    try:
        for name, df in tables.items():
            con.register(f"batch_{name}", df)

        con.execute("INSERT OR REPLACE INTO entities BY NAME SELECT * FROM batch_entities")
        for name in ["contracts", *CHILD_TABLES]:
            con.execute(f"DELETE FROM {name} WHERE contract_id IN (SELECT contract_id FROM batch_contracts)")
            con.execute(f"INSERT INTO {name} BY NAME SELECT * FROM batch_{name}")
        con.execute("COMMIT")
    except Exception:
        con.execute("ROLLBACK")
        raise
    finally:
        for name in tables:
            con.unregister(f"batch_{name}")


def row_counts(con):
    names = ["contracts", "entities", *CHILD_TABLES]
    return {n: con.sql(f"SELECT COUNT(*) FROM {n}").fetchall()[0][0] for n in names}


upsert(con, tables)
row_counts(con)

{'contracts': 1327,
 'entities': 966,
 'contract_parties': 4045,
 'contract_cpvs': 1328,
 'contract_locations': 1438}

In [8]:
# Idempotency check: loading the same batch a second time must not change anything
before = row_counts(con)
upsert(con, tables)
assert row_counts(con) == before, "re-running the load changed the row counts"
print("re-load OK, no duplicates:", before)

re-load OK, no duplicates: {'contracts': 1327, 'entities': 966, 'contract_parties': 4045, 'contract_cpvs': 1328, 'contract_locations': 1438}


## 4. Sanity checks
A few queries against the stored data. They also show the schema can answer the questions the project needs: concentration between the same authority and contractor, the procedure mix, and single bidding.

In [9]:
con.sql("""
    SELECT MIN(publication_date) AS first_published, MAX(publication_date) AS last_published,
           COUNT(*) AS contracts, SUM(contract_price) AS total_value,
           COUNT(*) FILTER (WHERE contract_price IS NULL) AS missing_price
    FROM contracts
""").df()

,first_published,last_published,contracts,total_value,missing_price
0,2026-01-07,2026-10-16,1327,79766666.01,0


In [10]:
# Which authority -> contractor pairs account for the most money? (concentration / favouritism signal)
con.sql("""
    SELECT authority, contractor, COUNT(*) AS contracts, SUM(contract_price) AS total_value
    FROM v_contracts
    GROUP BY ALL
    ORDER BY total_value DESC
    LIMIT 10
""").df()

,authority,contractor,contracts,total_value
0,Município da Covilhã,I-Sete Inovação Soluções Económicas e Tecnolog...,1,15479500.00
1,"Unidade Local de Saúde da Cova da Beira, EPE",PHILIPS PORTUGUESA S.A.,4,3347513.13
2,Município da Covilhã,"EDP Comercial, SA",1,2443127.17
3,"Unidade Local de Saúde da Cova da Beira, EPE",Serviço de Utilização Comum dos Hospitais,9,2354978.24
4,"Unidade Local de Saúde da Cova da Beira, EPE","JANSSEN CILAG FARMACEUTICA, LDA.",12,2334429.45
5,"Unidade Local de Saúde da Cova da Beira, EPE","ASTRAZENECA-PRODUTOS FARMACÊUTICOS,LDA.",10,1991127.96
6,"Unidade Local de Saúde da Cova da Beira, EPE",MERCK SHARP DOHME,6,1494447.12
7,"Unidade Local de Saúde da Cova da Beira, EPE",GILEAD SCIENCES LDA.,5,1367401.27
8,"Unidade Local de Saúde da Cova da Beira, EPE","ROCHE FARMACEUTICA QUIMICA, LDA",2,1286267.31
9,Município da Covilhã,"G9 TELECOM, S.A.",2,1156104.30


In [11]:
# Procedure mix, and how many procedures received only one bid
# (a single bid is expected for direct awards; it's a red flag only for competitive procedures like Concurso público)
con.sql("""
    WITH bids AS (
        SELECT contract_id, COUNT(*) AS bidders
        FROM contract_parties WHERE role = 'competitor' GROUP BY contract_id
    )
    SELECT c.procedure_type,
           COUNT(*)                                   AS contracts,
           SUM(c.contract_price)                      AS total_value,
           ROUND(AVG(b.bidders), 1)                   AS avg_bidders,
           COUNT(*) FILTER (WHERE b.bidders = 1)      AS single_bid
    FROM contracts c LEFT JOIN bids b USING (contract_id)
    GROUP BY ALL
    ORDER BY contracts DESC
""").df()

,procedure_type,contracts,total_value,avg_bidders,single_bid
0,Ajuste Direto Regime Geral,431,12728712.62,1.0,61
1,Ao abrigo de acordo-quadro (art.º 259.º),425,14384602.74,5.8,4
2,Concurso público,245,41452268.51,6.3,9
3,Consulta Prévia,146,3850128.49,1.7,70
4,Contratação excluída II,50,3574546.02,1.0,50
5,Ao abrigo de acordo-quadro (art.º 258.º),15,1060327.26,1.0,2
6,Consulta Prévia Simplificada,9,1987867.41,1.9,3
7,Concurso limitado por prévia qualificação,4,143744.96,NaN,0
8,Setores especiais – isenção parte II,2,584468.00,4.5,0


In [12]:
# One entity, many spellings: confirms that keying on NIF was needed
con.sql("""
    SELECT p.nif, COUNT(DISTINCT json_extract_string(c.raw_record, '$.adjudicante[0]')) AS spellings_in_raw
    FROM contract_parties p JOIN contracts c USING (contract_id)
    WHERE p.role = 'authority'
    GROUP BY p.nif HAVING spellings_in_raw > 1
    ORDER BY spellings_in_raw DESC
""").df().merge(con.sql("SELECT * FROM entities").df(), on="nif")

,nif,spellings_in_raw,name
0,501442600,6,Centro de Emprego e Formação Profissional de Beja
1,600031691,2,Escola Profissional Agrícola Quinta da Lageosa...
2,600017982,2,Serviços de Ação Social da Universidade da Bei...
3,504647040,2,Associação para a Escola Profissional de Artes...
4,506361659,2,"Unidade Local de Saúde da Cova da Beira, EPE"
5,509906478,2,Centro de Formação Profissional da Indústria T...
6,507611977,2,"ADC - Águas da Covilhã, EM"
7,502567864,2,Centro Social e Cultural de Verdelhos


In [13]:
con.close()
push_db()   # Colab: copy the database back to Drive

database saved to Drive (5.5 MB)
